<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: Anita Hovsepian  
Github Username:anitahov   
USC ID: 6881664760


## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [27]:
import os
import re
import glob
import zipfile
import io
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RANDOM_STATE = 552
rng = np.random.default_rng(RANDOM_STATE)


Get the AReM Data Set

In [28]:
import pandas as pd
import re

BASE_URL = "https://raw.githubusercontent.com/anitahov/DSCI-552-HW3/main/AReM"

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

dataset_counts = {
    "bending1": 7,
    "bending2": 6,
    "cycling": 15,
    "lying": 15,
    "sitting": 15,
    "standing": 15,
    "walking": 15
}

files = []

for activity in activities:
    for dataset_num in range(1, dataset_counts[activity] + 1):
        files.append(
            f"{BASE_URL}/{activity}/dataset{dataset_num}.csv"
        )

print("Number of instances:", len(files))
print("Activities:", activities)

Number of instances: 88
Activities: ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']


### (b) Test and Train Data

In [29]:
def dataset_number(path):
    return int(re.search(r'dataset(\d+)\.csv$', path).group(1))

def activity_name(path):
    return os.path.basename(os.path.dirname(path))

def is_test_file(path):
    activity = activity_name(path)
    n = dataset_number(path)
    if activity in {'bending1', 'bending2'}:
        return n in {1, 2}
    return n in {1, 2, 3}

test_files = sorted([f for f in files if is_test_file(f)], key=lambda x: (activity_name(x), dataset_number(x)))
train_files = sorted([f for f in files if not is_test_file(f)], key=lambda x: (activity_name(x), dataset_number(x)))

print('Training instances:', len(train_files))
print('Test instances:', len(test_files))
print('Total:', len(train_files) + len(test_files))
print()
print('Test split by activity:')
print(pd.Series([activity_name(f) for f in test_files]).value_counts().sort_index())


Training instances: 69
Test instances: 19
Total: 88

Test split by activity:
bending1    2
bending2    2
cycling     3
lying       3
sitting     3
standing    3
walking     3
Name: count, dtype: int64


### (c) Feature Extraction

#### i. Research

Common **time-domain features** used for time-series classification include minimum, maximum, mean, median, standard deviation, variance, range, first quartile (25th percentile), third quartile (75th percentile), interquartile range, root mean square (RMS), energy, skewness, kurtosis, zero-crossing rate, mean/median absolute deviation, and selected percentiles. Depending on the application, temporal features such as slope, autocorrelation, and peak-related statistics can also be useful.

For this homework, the required seven features are minimum, maximum, mean, median, standard deviation, first quartile, and third quartile. They will be extracted separately from each of the six sensor time series in every instance.


#### ii. Extraction

In [30]:
import pandas as pd

url = "https://raw.githubusercontent.com/anitahov/DSCI-552-HW3/main/AReM/bending1/dataset1.csv"

df = pd.read_csv(
    url,
    skiprows=4,
    header=None
)

display(df.head())

,0,1,2,3,4,5,6
0,# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
1,0,39.25,0.43,22.75,0.43,33.75,1.30
2,250,39.25,0.43,23.00,0.00,33.00,0.00
3,500,39.25,0.43,23.25,0.43,33.00,0.00
4,750,39.50,0.50,23.00,0.71,33.00,0.00


#### iii. Standard Deviation

In [26]:
# (c)(ii) Extract time-domain features from all 88 AReM instances

import pandas as pd
import numpy as np
import io
import urllib.request
import re

# Six time series in each AReM dataset
sensor_names = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

# Seven required time-domain features
stat_names = [
    "min",
    "max",
    "mean",
    "median",
    "std",
    "1st_quart",
    "3rd_quart"
]


# ------------------------------------------------------------
# Read and clean an AReM CSV file from GitHub
# ------------------------------------------------------------

def read_arem_csv(url):

    # Download the file
    with urllib.request.urlopen(url) as response:
        text = response.read().decode("utf-8", errors="ignore")

    # Split into lines
    lines = text.splitlines()

    # Remove the first four metadata lines
    lines = lines[4:]

    cleaned_lines = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Skip the column-name line
        if line.startswith("#"):
            continue

        # Remove trailing commas from malformed rows
        line = line.rstrip(",")

        cleaned_lines.append(line)

    # Rebuild cleaned CSV text
    cleaned_text = "\n".join(cleaned_lines)

    # Read CSV
    df = pd.read_csv(
        io.StringIO(cleaned_text),
        header=None,
        sep=r"\s+|,",
        engine="python"
    )

    # Keep exactly 7 columns:
    # time + six sensor time series
    df = df.iloc[:, :7].copy()

    df.columns = ["time"] + sensor_names

    # Convert sensor columns to numeric
    for col in sensor_names:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # Remove rows with invalid sensor values
    df = df.dropna(subset=sensor_names)

    return df


# ------------------------------------------------------------
# Extract 7 features from each of the 6 time series
# ------------------------------------------------------------

def extract_features(url):

    df = read_arem_csv(url)

    activity = activity_name(url)
    dataset = dataset_number(url)

    row = {
        "activity": activity,
        "dataset": dataset,
        "split": "test" if is_test_file(url) else "train"
    }

    for i, col in enumerate(sensor_names, start=1):

        series = df[col]

        row[f"min{i}"] = series.min()
        row[f"max{i}"] = series.max()
        row[f"mean{i}"] = series.mean()
        row[f"median{i}"] = series.median()
        row[f"std{i}"] = series.std(ddof=1)
        row[f"1st_quart{i}"] = series.quantile(0.25)
        row[f"3rd_quart{i}"] = series.quantile(0.75)

    return row


# ------------------------------------------------------------
# Sort all 88 files
# ------------------------------------------------------------

all_files = sorted(
    files,
    key=lambda x: (
        activity_name(x),
        dataset_number(x)
    )
)

print("Total instances:", len(all_files))


# ------------------------------------------------------------
# Extract features from all 88 instances
# ------------------------------------------------------------

rows = []

for count, url in enumerate(all_files, start=1):

    activity = activity_name(url)
    dataset = dataset_number(url)

    print(
        f"Processing {count}/88: "
        f"{activity} dataset{dataset}"
    )

    row = extract_features(url)

    rows.append(row)


# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

feature_df = pd.DataFrame(rows)


# Create names for all 42 extracted features
feature_cols = [
    f"{stat}{i}"
    for i in range(1, 7)
    for stat in stat_names
]


# Arrange columns
feature_df = feature_df[
    ["activity", "dataset", "split"] + feature_cols
]


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nFeature data shape:", feature_df.shape)

print(
    "Number of extracted features:",
    len(feature_cols)
)

print("\nTrain/Test:")
print(
    feature_df["split"].value_counts()
)

display(feature_df)


# ------------------------------------------------------------
# Save extracted features
# ------------------------------------------------------------

feature_df.to_csv(
    "AReM_extracted_features.csv",
    index=False
)

print(
    "\nSaved: AReM_extracted_features.csv"
)

Total instances: 88
Processing 1/88: bending1 dataset1
Processing 2/88: bending1 dataset2
Processing 3/88: bending1 dataset3
Processing 4/88: bending1 dataset4
Processing 5/88: bending1 dataset5
Processing 6/88: bending1 dataset6
Processing 7/88: bending1 dataset7
Processing 8/88: bending2 dataset1
Processing 9/88: bending2 dataset2
Processing 10/88: bending2 dataset3
Processing 11/88: bending2 dataset4
Processing 12/88: bending2 dataset5
Processing 13/88: bending2 dataset6
Processing 14/88: cycling dataset1
Processing 15/88: cycling dataset2
Processing 16/88: cycling dataset3
Processing 17/88: cycling dataset4
Processing 18/88: cycling dataset5
Processing 19/88: cycling dataset6
Processing 20/88: cycling dataset7
Processing 21/88: cycling dataset8
Processing 22/88: cycling dataset9
Processing 23/88: cycling dataset10
Processing 24/88: cycling dataset11
Processing 25/88: cycling dataset12
Processing 26/88: cycling dataset13
Processing 27/88: cycling dataset14
Processing 28/88: cycling 

,activity,dataset,split,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,0.430,0.322605,0.00,0.5000,4.00,29.50,19.040937,19.250,4.462952,16.00,23.2500,0.0,7.23,0.832542,0.50,0.965659,0.0000,1.1200,27.25,38.25,34.311292,35.00,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,0.470,0.289158,0.00,0.5000,2.00,29.50,20.096979,21.000,3.893737,19.00,22.2500,0.0,5.76,0.876438,0.50,1.047259,0.0000,1.1450,27.67,38.50,33.024583,33.00,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,0.470,0.338690,0.00,0.5000,6.50,29.75,22.122354,23.000,3.030943,19.75,24.0000,0.0,4.44,0.497312,0.43,0.550657,0.0000,0.8300,29.00,38.50,35.588458,36.00,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,0.500,0.630860,0.00,1.1200,8.50,30.00,22.183625,23.000,3.810469,20.50,24.3725,0.0,5.15,0.989917,0.83,0.953730,0.4300,1.3000,20.00,38.67,33.493917,35.00,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,0.500,0.405469,0.43,0.7100,3.00,28.25,19.006563,19.125,4.087107,16.50,22.0625,0.0,6.42,0.841875,0.50,0.928801,0.4300,1.1200,23.67,37.50,29.857083,30.00,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking,11,train,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,4.260,2.461077,2.87,6.1050,7.33,23.25,15.317854,15.330,2.970440,13.25,17.2700,0.0,9.00,3.227229,3.10,1.647474,2.0500,4.2600,6.33,25.00,15.892146,16.00,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,walking,12,train,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,3.900,2.599284,2.49,5.9700,6.25,22.25,15.209771,15.250,2.711022,13.50,17.0000,0.0,9.00,3.277229,3.11,1.500046,2.1700,4.3900,6.25,23.25,15.550458,15.75,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,walking,13,train,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,4.135,2.360448,2.49,5.8000,7.00,22.67,15.185729,15.250,2.757604,13.25,17.0000,0.0,8.34,3.028687,2.86,1.665184,1.7900,4.1100,7.00,22.75,15.759542,15.75,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,walking,14,train,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,4.380583,3.925,2.435437,2.50,5.9325,6.67,24.25,15.471188,15.500,2.813077,13.50,17.2500,0.0,9.90,3.205688,3.03,1.519348,2.0575,4.1900,5.50,23.50,15.873896,16.00,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000



Saved: AReM_extracted_features.csv


#### iv. Select Features

I would select minimum, mean, and maximum as the three most important time-domain features for this first-stage representation.

- **Minimum** captures the lower extreme of the sensor signal and can help distinguish activities with different low-end movement levels.
- **Mean** summarizes the central level of the signal over the complete activity instance and is generally stable and interpretable.
- **Maximum** captures the upper extreme and complements the minimum by describing the signal's overall amplitude range.

Together, these three features provide a compact description of the location and extremes of each time series. Standard deviation and quartiles are also useful, but min/mean/max provide a simple, nonredundant starting set for the later classification task.


## 2. ISLR 3.7.4

### (a) Linear Train

Let the true relationship between $X$ and $Y$ be linear. Compare a least-squares linear regression with a cubic regression that includes additional polynomial terms.

The training RSS for the cubic regression will be less than or equal to the training RSS for the linear regression. The cubic model contains the linear model as a special case, so least squares can set the coefficients of the extra polynomial terms to zero if they do not help. Therefore, adding predictors cannot increase the minimum training RSS. In practice, it will usually be slightly lower because the extra flexibility can fit some training noise.


### (b) Linear Test

For the test RSS, the linear regression is expected to perform better when the true relationship is actually linear. The cubic model has unnecessary flexibility and can fit noise in the training sample, increasing variance without reducing bias in a useful way. Thus, we would generally expect the linear regression to have the lower test RSS, although a particular random test sample could produce a different result.


### (c) Not Linear Train

Now suppose the true relationship is not linear, but we do not know how far it is from linear. On the training data, the cubic regression will still have training RSS less than or equal to that of the linear regression. This follows from model nesting: the cubic model has all terms available to the linear model plus additional terms, so it can always reproduce the linear fit and may improve on it.


### (d) Not Linear Testing

For the test RSS when the true relationship is nonlinear, there is not enough information to determine which model will be lower. If the true relationship is well approximated by a cubic polynomial, the cubic model may reduce bias enough to outperform the linear model. If the nonlinearity is weak, or if the true function is not well represented by a cubic form, the additional flexibility can increase variance and the linear model may perform better. The result depends on the unknown true relationship, noise level, and sample.


## 3. ISLR 3.7.3 - Extra Practice

#### (a)

The estimated salary model is:

$$
\hat{Y} = 50 + 20(\text{GPA}) + 0.07(\text{IQ}) + 35(\text{Gender}) + 0.01(\text{GPA}\times\text{IQ}) - 10(\text{GPA}\times\text{Gender})
$$

where Gender = 1 for female and Gender = 0 for male.

For a female, the effect relative to a male with the same GPA and IQ is:

$$
35 - 10(\text{GPA})
$$

If GPA is greater than 3.5, this quantity is negative. Therefore, for a sufficiently high GPA, males earn more on average than females.

The correct answer is (iii).

#### (b)

For a female with IQ = 110 and GPA = 4.0:

$$
\hat{Y} = 50 + 20(4) + 0.07(110) + 35 + 0.01(4)(110) - 10(4)
$$

$$
\hat{Y} = 50 + 80 + 7.7 + 35 + 4.4 - 40
$$

$$
\hat{Y} = 137.1
$$

Since salary is measured in thousands of dollars, the predicted salary is:

$$
\boxed{\$137,100}
$$

#### (c)

False. The fact that the coefficient for the GPA/IQ interaction is small does not by itself show that there is little evidence of an interaction effect. Statistical significance depends on the coefficient relative to its standard error, usually evaluated using a t-statistic and p-value. The scale of the variables also affects the numerical size of a regression coefficient.

## 3. ISLR 3.7.5 - Extra Practice

For simple linear regression without an intercept, the model is:

$$
Y_i = \beta X_i + \epsilon_i
$$

The least-squares estimate of $\beta$ is:

$$
\hat{\beta} =
\frac{\sum_{i'=1}^{n} x_{i'}y_{i'}}
{\sum_{j=1}^{n} x_j^2}
$$

The fitted value for observation $i$ is:

$$
\hat{y}_i = x_i\hat{\beta}
$$

Substituting the expression for $\hat{\beta}$ gives:

$$
\hat{y}_i =
x_i
\frac{\sum_{i'=1}^{n}x_{i'}y_{i'}}
{\sum_{j=1}^{n}x_j^2}
$$

This can be rewritten as:

$$
\hat{y}_i =
\sum_{i'=1}^{n}
\frac{x_i x_{i'}}
{\sum_{j=1}^{n}x_j^2}
y_{i'}
$$

Therefore,

$$
\hat{y}_i =
\sum_{i'=1}^{n} a_{i'}y_{i'}
$$

where

$$
\boxed{
a_{i'} =
\frac{x_i x_{i'}}
{\sum_{j=1}^{n}x_j^2}
}
$$

Thus, each fitted value $\hat{y}_i$ is a linear combination of the observed response values $y_1,\ldots,y_n$, with weights determined entirely by the predictor values.